# 02 — Base Model Baseline (NO fine-tuning)

Phase 3-4. **This must run before any training.**

The pretrained model is loaded with no adapter, queried with three frozen
prompts, and every raw response is saved. The evaluation subset chosen here
is frozen and reused by the fine-tuned evaluation.


In [ ]:
# --- Colab setup (skip if running locally) ---
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('casting-defect-vlm'):
        # Replace with your repository URL, or upload the folder to Colab.
        raise SystemExit('Upload the casting-defect-vlm project folder to Colab first.')
    %cd casting-defect-vlm
    !pip install -q -r requirements.txt

sys.path.insert(0, os.path.abspath('..' if os.path.basename(os.getcwd())=='notebooks' else '.'))
print('python', sys.version.split()[0], '| colab:', IN_COLAB)


In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('WARNING: no CUDA GPU. Baseline and training need one.')


## 1. Prepare splits first

Verify the grouping heuristic before trusting any split.


In [ ]:
!python scripts/prepare_training_data.py --inspect-groups


> If every group has exactly 1 image, the heuristic did not match this
> dataset's filenames. Fix `derive_group_key()` in `src/dataset.py` before
> continuing — otherwise the leakage guarantee does not hold.


In [ ]:
!python scripts/prepare_training_data.py


## 2. The frozen prompts


In [ ]:
from src.prompts import EVAL_PROMPTS
for pid, text in EVAL_PROMPTS.items():
    print('='*60); print(pid); print('='*60); print(text); print()


## 3. Smoke test (5 images)


In [ ]:
!python scripts/run_baseline.py --limit 5


## 4. Full baseline run


In [ ]:
!python scripts/run_baseline.py


## 5. Results


In [ ]:
import json, pandas as pd
from pathlib import Path

res = pd.read_csv('results/baseline/baseline_results.csv')
print('rows:', len(res))
res[['image_id','ground_truth','prompt_id','parsed_prediction','confidence','correct']].head(12)


## 6. Metrics


## 6. TRACK A (PRIMARY) — binary OK vs Defective

Reported on the frozen balanced slice (12 OK + 12 Defective), with the
full 178-image test set alongside for completeness.


In [ ]:
import json, pandas as pd
from pathlib import Path

m = json.loads(Path('results/baseline/baseline_metrics.json').read_text())
a = m['track_a_binary']
bal, full = a['balanced_subset'], a['full_test_set']

rows = []
for name, d in [('balanced slice', bal), ('full test set', full)]:
    rows.append({
        'view': name,
        'accuracy': d.get('accuracy'),
        'balanced_acc': d.get('balanced_accuracy'),
        'precision_def': d.get('precision_defective'),
        'recall_def': d.get('recall_defective'),
        'f1_def': d.get('f1_defective'),
        'macro_f1': d.get('macro_f1'),
        'ok_recall': d.get('ok_recall'),
    })
pd.DataFrame(rows).set_index('view')


### Confusion matrix and small-sample caveat


In [ ]:
cm = bal['confusion_matrix']
print('BALANCED SLICE')
print('  TN (OK->OK)              :', cm['true_negative_ok_as_ok'])
print('  FP (OK->Defective)       :', cm['false_positive_ok_as_defective'])
print('  FN (Defective->OK)       :', cm['false_negative_defective_as_ok'])
print('  TP (Defective->Defective):', cm['true_positive_defective_as_defective'])
print()
print('OK recall       :', bal['ok_recall'], '95% CI', bal['ok_recall_95ci'])
print('Defective recall:', bal['defective_recall'], '95% CI', bal['defective_recall_95ci'])
print()
print(bal.get('small_sample_warning'))
print(bal.get('ci_basis'))


In [ ]:
from IPython.display import Image, display
for f in ['baseline_confusion_matrix_balanced.png', 'baseline_confusion_matrix.png']:
    p = Path('results/baseline') / f
    if p.exists():
        print(f); display(Image(str(p)))


## 7. TRACK B (SECONDARY) — 12-class defect type

**Coverage matters here.** The prompts never showed the model the dataset's
class vocabulary, so many responses will not map to one of the 12 labels.
Metrics are computed only on those that do — a self-selected, optimistically
biased subset. Always read them next to coverage.


In [ ]:
b = m['track_b_defect_type']['full_test_set']
print('coverage       :', b.get('coverage'),
      f"({b.get('n_valid_class_predictions')}/{b.get('n_total')} valid)")
print('unmatched      :', b.get('n_unmatched'))
for k in ['accuracy','macro_precision','macro_recall','macro_f1',
          'weighted_precision','weighted_recall','weighted_f1']:
    print(f'{k:<20}', b.get(k))
print()
print(b.get('classification_report_text', 'n/a'))


In [ ]:
p = Path('results/baseline/baseline_defect_type_confusion_matrix.png')
display(Image(str(p))) if p.exists() else print('not generated')


## 8. Raw outputs

Every response is preserved verbatim — your professor asked specifically for
the base-model queries and outcomes to be documented.


In [ ]:
with open('results/baseline/baseline_raw_outputs.jsonl') as fh:
    raws = [json.loads(l) for l in fh]
print('raw records:', len(raws))
for r in raws[:5]:
    print('='*72)
    print(r['relpath'], '| truth:', r['ground_truth'], '| prompt:', r['prompt_id'])
    print(r['raw_response'])
    print('-> parsed:', r['parsed_prediction'], '| class:', r.get('predicted_class'),
          '| correct:', r['correct'])


## 9. Error analysis


In [ ]:
errs = pd.read_csv('results/baseline/error_analysis.csv')
print(errs['error_type'].value_counts(), '\n')
print(errs['failure_flags'].value_counts())


### Missed defects (said OK for a Defective casting)


In [ ]:
res = pd.read_csv('results/baseline/baseline_results.csv')
fn = res[(res.ground_truth=='Defective') & (res.parsed_prediction=='OK')]
print(len(fn), 'false negatives')
fn[['relpath','true_defect_type','prompt_id','confidence']].head(10)


### False alarms (said Defective for an OK casting)


In [ ]:
fp = res[(res.ground_truth=='OK') & (res.parsed_prediction=='Defective')]
print(len(fp), 'false positives')
fp[['relpath','prompt_id','predicted_defect_type','confidence']].head(10)


### Hallucinated defect types on OK castings


In [ ]:
h = res[(res.ground_truth=='OK') & (~res.predicted_defect_type.astype(str).isin(['None','Unknown','nan']))]
print(len(h), 'hallucinated defect types')
h[['relpath','prompt_id','predicted_defect_type','predicted_class','confidence']].head(10)


## 10. Generate the report


In [ ]:
!python scripts/generate_baseline_report.py


## 11. Observations

_Record what the base model actually did. This is the evidence for whether
fine-tuning is needed._

> Confidence figures are **model-reported**, not calibrated probabilities.
> Track B measures recognition of **synthetic** defect textures.
